In [1]:
import numpy as np

In [ ]:
def softmax(x):
    x = x - np.max(x, axis=-1, keepdims=True) # for numerical stability
    e_x = np.exp(x)
    return e_x / np.sum(e_x, axis=-1, keepdims=True)

In [ ]:
def cross_entropy_loss(y_true, y_pred):
    m = y_true.shape[0]
    p = softmax(y_pred)
    log_likelihood = -np.log(p[range(m), y_true])
    loss = np.sum(log_likelihood) / m
    return loss

In [ ]:
def sigmoid(x):
    if x >= 0:
        return 1 / (1 + np.exp(-x))
    else:
        exp_x = np.exp(x)
        return exp_x / (1 + exp_x)

In [ ]:
def attention(query, key, value):
    d_k = query.shape[-1]
    scores = np.dot(query, key.T) / np.sqrt(d_k)
    attn_weights = softmax(scores)
    output = np.dot(attn_weights, value)
    return output


In [ ]:
def backward(y_true, y_pred): # for softmax + cross-entropy
    m = y_true.shape[0]
    p = softmax(y_pred)
    p[range(m), y_true] -= 1
    return p / m

In [ ]:
def linear_backward(dout, cache): # dout is dL/dout, cache is (x, w, b)
    x, w, b = cache
    
    dw = np.dot(x.T, dout) # dL/dw = x^T * dL/dout
    db = np.sum(dout, axis=0) # dL/db = sum of dL/dout over all samples
    dx = np.dot(dout, w.T)  # dL/dx = dL/dout * w^T
    
    return dx, dw, db # return gradients with respect to x, w, b

In [ ]:
def multi_head_attention(query, key, value, num_heads):
    d_k = query.shape[-1] // num_heads
    heads = []
    for i in range(num_heads):
        q = query[:, i*d_k:(i+1)*d_k]
        k = key[:, i*d_k:(i+1)*d_k]
        v = value[:, i*d_k:(i+1)*d_k]
        head = attention(q, k, v)
        heads.append(head)
    return np.concatenate(heads, axis=-1)


In [ ]:
def layerNorm(x, gamma, beta, eps=1e-5):
    mean = np.mean(x, axis=-1, keepdims=True)
    var = np.var(x, axis=-1, keepdims=True)
    x_normalized = (x - mean) / np.sqrt(var + eps)
    return gamma * x_normalized + beta

: 